# Event displays + per-sensor likelihood vs observed time

Thin wrapper over `per_sensor_time_overlay.py` and `per_sensor_p_first_validation.py`.

For one ground-truth event:

1. **Event displays** — 2D unwrapped detector views (true / pred × charge / time).
2. **Per-sensor diagnostics** — three-panel plots for representative sensors:
   - λ(t) (Gaussian-convolved per-photon arrival rate)
   - cumulative Λ(t) with 1-PE reference
   - first-arrival density `p_first(t) = λ(t)·exp(-Λ(t))` with the observed t_obs overlaid and per-sensor NLL annotated.
3. **Empirical validation** — overlay the empirical t_obs histogram (across many fresh data-sim trials on the same true event) against the analytical `p_first(t)`.

Heavy logic lives in the two scripts. Each cell imports and calls in.


## Setup


In [ ]:
import sys, time
sys.path.append('..')
import jax, jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

import per_sensor_time_overlay as pso
import per_sensor_p_first_validation as psv

# Knobs (override the script defaults here)
pso.ENTRY_IDX = 2
pso.SEED = 44
pso.N_PER_STRATUM = 3
psv.N_TRIALS = 200          # number of data-sim re-samples for validation
print(f'entry={pso.ENTRY_IDX}, seed={pso.SEED}, N_PER_STRATUM={pso.N_PER_STRATUM}, '
      f'σ_TTS={pso.SIGMA_TTS_NS} ns, N_TRIALS={psv.N_TRIALS}')


## Build simulators and one event


In [ ]:
detector, num_detectors, data_sim, pred_sim = pso.build_simulators()
print(f'Detectors: {num_detectors}')

t0 = time.time()
(true_track, true_data, pred_data_pp,
 pos, direction, energy, photon_data) = pso.build_event(
    detector, data_sim, pred_sim, jax.random.PRNGKey(pso.SEED))
print(f'build_event: {time.time() - t0:.1f}s')

# Stay in JAX. The simulator returns device arrays; downstream JIT'd helpers
# accept them directly (no host transfer). We convert to numpy lazily only at
# matplotlib boundaries.
log_w_j, flat_times_j, flat_indices_j, total_charge_j = pred_data_pp
weights_j = jnp.exp(log_w_j)

# Just one host pull for the print + sensor stratification (small)
hit_counts0 = np.asarray(true_data[0])
n_hit = int(np.sum(hit_counts0 > 0))
print(f'event: E = {float(energy):.1f} MeV, hit sensors = {n_hit}/{num_detectors}')


## 1. Build per-sensor analytical diagnostics (one JIT call)

`evaluate` is `@jax.jit`'d. We pass JAX arrays directly so the call has no
host→device copy. The returned `diag` is a dict of JAX arrays; we lazily pull
to numpy only when matplotlib needs them.


In [ ]:
print('Computing analytical p_first diagnostics ...')
t0 = time.time()
evaluate = pso.make_diagnostics_evaluator(
    num_sensors=num_detectors, n_grid=200, sigma_tts=pso.SIGMA_TTS_NS)
diag = evaluate(flat_times_j, weights_j, flat_indices_j, true_data[1])
print(f'  done in {time.time() - t0:.2f}s (grid: {diag["t_grid"].shape[1]} samples/sensor)')


## 2. Event displays — 4 unwrapped 2D views


In [ ]:
pso.EVENT_DISPLAY_DIR.mkdir(parents=True, exist_ok=True)
# matplotlib boundary: pull the per-sensor (NUM_DETECTORS-element) arrays
# we actually plot. The big per-photon arrays stay on-device.
pred_charges = np.asarray(total_charge_j)
mode_np   = np.asarray(diag['mode'])
t_lead_np = np.asarray(diag['t_lead'])
pred_times = np.where(np.isfinite(mode_np), mode_np, t_lead_np)
pred_times = np.nan_to_num(pred_times, nan=0.0)
pso.render_event_displays(true_data, pred_charges, pred_times)
display_paths = sorted(pso.EVENT_DISPLAY_DIR.glob('*.png'))
print(f'saved {len(display_paths)} displays in {pso.EVENT_DISPLAY_DIR}/')
from IPython.display import Image, display
for p in display_paths[-4:]:
    print(p.name)
    display(Image(filename=str(p)))


## 3. Per-sensor 3-panel plots

Pick high / medium / low PE sensors and plot:
  (a) λ(t),  (b) cumulative Λ(t),  (c) p_first(t) = λ(t)·exp(-Λ(t)) with t_obs overlaid.


In [ ]:
# Stratification works on per-sensor arrays (NUM_DETECTORS elements — small).
total_pe = np.asarray(total_charge_j)
high, medium, low = pso.stratified_sensor_selection(
    total_pe, hit_counts0, n_per_stratum=pso.N_PER_STRATUM, hit_threshold=1.0)
print(f'High PE   sensors: {high}')
print(f'Medium PE sensors: {medium}')
print(f'Low PE    sensors: {low}')

# matplotlib boundary for the per-photon arrays — needed for plot_single_sensor's
# per-sensor masking. (Plotting could be made fully JAX-native by refactoring
# plot_single_sensor to mask sensors in jax and only pull each sensor's small
# subset; left as future work in the script.)
flat_times   = np.asarray(flat_times_j)
flat_indices = np.asarray(flat_indices_j)
weights      = np.asarray(weights_j)

pso.PER_SENSOR_DIR.mkdir(parents=True, exist_ok=True)
pso.render_per_sensor_plots(
    flat_times=flat_times, weights=weights, flat_indices=flat_indices,
    total_pe=total_pe, true_data=true_data,
    stratum_sensors={'high': high, 'medium': medium, 'low': low},
    diag=diag,
)
sensor_paths = sorted(pso.PER_SENSOR_DIR.glob('*.png'))
print(f'saved {len(sensor_paths)} per-sensor plots in {pso.PER_SENSOR_DIR}/')
# Show a few inline (one per stratum)
shown_strata = set()
for p in sensor_paths:
    stratum = p.stem.split('_')[0]
    if stratum in shown_strata:
        continue
    shown_strata.add(stratum)
    print(p.name)
    display(Image(filename=str(p)))
    if len(shown_strata) >= 3:
        break


## 4. Empirical-vs-analytical p_first validation

Run the data simulator MANY times on the same true event with fresh keys.
Overlay the empirical t_obs histogram (across trials, conditioned on hit) on
the analytical p_first(t) used by the loss. They should agree modulo MC noise.

`diag` was already computed in section 1 — we reuse it.


In [ ]:
print(f'Running {psv.N_TRIALS} data trials ...')
t0 = time.time()
all_hit_counts, all_hit_times = psv.run_data_trials(
    data_sim, true_track, photon_data, jax.random.PRNGKey(pso.SEED + 1),
    psv.N_TRIALS)
print(f'  done: {time.time() - t0:.2f}s')


In [ ]:
# Plot validation for a representative sensor in each stratum
psv.OUT_DIR.mkdir(parents=True, exist_ok=True)
label_map = {'high': 'High PE', 'medium': 'Medium PE', 'low': 'Low PE'}
selected = {'high': high[:1], 'medium': medium[:1], 'low': low[:1]}
val_paths = []
for stratum_key, sensors in selected.items():
    for s in sensors:
        out = psv.OUT_DIR / f'{stratum_key}_sensor_{s}.png'
        psv.plot_validation(
            sensor_id=s, stratum_label=label_map[stratum_key],
            all_hit_counts=all_hit_counts, all_hit_times=all_hit_times,
            diag=diag, out_path=str(out))
        val_paths.append(out)
print(f'saved {len(val_paths)} validation plots in {psv.OUT_DIR}/')
for p in val_paths:
    print(p.name)
    display(Image(filename=str(p)))


## Notes

- Knobs at the top: `pso.ENTRY_IDX`, `pso.SEED`, `pso.N_PER_STRATUM`, `psv.N_TRIALS`.
  For other settings (temperature, K, σ_TTS, etc.) edit
  `per_sensor_time_overlay.py` directly — those are module-level constants.
- All figures are saved to `figures/event_displays/`, `figures/per_sensor_time/`,
  and `figures/per_sensor_validation/`.
- The validation cell takes ~1–2 min for `N_TRIALS=200` because it runs the
  data simulator N_TRIALS times.

### JAX / numpy convention used in this notebook

- The per-photon arrays from `pred_sim` stay on-device as JAX arrays
  (`flat_times_j`, `flat_indices_j`, `weights_j`, `total_charge_j`).
- The JIT'd `evaluate(...)` consumes them directly — no host transfer; XLA can
  fuse the diagnostic computation with the simulator output if running on GPU.
- `np.asarray(...)` is only invoked at matplotlib boundaries, and ideally on the
  small per-sensor (~10k-element) arrays rather than the big per-photon
  (~1M-element) ones. The per-photon pull in section 3 is the one remaining
  large transfer; a full refactor of `plot_single_sensor` to mask sensors in
  JAX could eliminate it, but it isn't worth it for a one-event diagnostic.
